# 00 环境自检 + 挂盘

# 00 · 环境自检 + 挂盘（每次开会话先跑这一个）
目的：30 秒内搞清楚「这台机器是什么卡、还剩多少空间、模型缓存放哪、产物往哪存」。
关键点：**把 HF 缓存和输出都放到 Drive**，这样断线重连不用重新下模型，产出也不会随会话蒸发。

In [ ]:
import os, sys, json, shutil, subprocess, textwrap, time
from pathlib import Path

def sh(cmd, check=True, quiet=False):
    """在 Colab 和本地都能用的 shell 执行器。"""
    if not quiet:
        print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout and not quiet:
        print(p.stdout.strip()[-4000:])
    if p.returncode != 0:
        print(p.stderr.strip()[-2000:])
        if check:
            raise RuntimeError(f"命令失败({p.returncode}): {cmd}")
    return p

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

In [ ]:
# ── 1. 机器体检 ──
report = {"in_colab": IN_COLAB, "python": sys.version.split()[0]}

try:
    import torch
    report["torch"] = torch.__version__
    report["cuda"] = torch.version.cuda
    report["gpu_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        report["gpu"] = p.name
        report["vram_gb"] = round(p.total_memory / 1024**3, 1)
        report["capability"] = f"{p.major}.{p.minor}"
except Exception as e:
    report["torch_error"] = str(e)

if shutil.which("nvidia-smi"):
    out = subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader",
                         shell=True, capture_output=True, text=True).stdout.strip()
    report["nvidia_smi"] = out

du = shutil.disk_usage("/content" if Path("/content").exists() else ".")
report["disk_free_gb"] = round(du.free / 1024**3, 1)
report["cpu_count"] = os.cpu_count()
try:
    report["ram_gb"] = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3, 1)
except Exception:
    report["ram_gb"] = None

print(json.dumps(report, ensure_ascii=False, indent=2))

if not report.get("gpu_available"):
    print("\n⚠️ 没有 GPU。右上角「连接」→「更改运行时类型」→ 硬件加速器选 T4 GPU，再重跑本格。")

In [ ]:
# ── 2. 挂 Google Drive（模型缓存 + 产物都放这）──
DRIVE = Path("/content/drive/MyDrive")
MOUNTED = False
if IN_COLAB:
    try:
        from google.colab import drive
        if not DRIVE.exists() or not any(DRIVE.iterdir()):
            drive.mount("/content/drive", force_remount=False)
        MOUNTED = DRIVE.exists()
    except Exception as e:
        print("挂载失败（可跳过，产物就只在 /content 里）:", e)
else:
    MOUNTED = False
print("Drive:", "已挂载 " + str(DRIVE) if MOUNTED else "未挂载")

In [ ]:
# ── 3. 目录约定：缓存放 Drive，产物放 Drive/outputs/<项目名>/ ──
PROJECT = os.environ.get("CK_PROJECT", "colabkit")
BASE = (DRIVE / "colabkit") if MOUNTED else Path("/content/colabkit")
PATHS = {
    "base": BASE,
    "hf_cache": BASE / "hf-cache",
    "outputs": BASE / "outputs" / PROJECT,
    "logs": BASE / "logs",
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(PATHS["hf_cache"])
os.environ["HUGGINGFACE_HUB_CACHE"] = str(PATHS["hf_cache"] / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(PATHS["hf_cache"] / "hub")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(json.dumps({k: str(v) for k, v in PATHS.items()}, ensure_ascii=False, indent=2))

In [ ]:
# ── 4. 基础依赖（只装缺的，装完不用重启）──
sh("pip -q install -U 'huggingface_hub[hf_transfer]' imageio imageio-ffmpeg soundfile librosa", check=False)
sh("ffmpeg -version | head -1", check=False)

try:
    du = shutil.disk_usage(str(BASE))
    print(f"工作盘剩余 {du.free/1024**3:.1f} GB")
    if du.free / 1024**3 < 10:
        print("⚠️ 剩余空间偏小：视频模型往往 5–15 GB，建议先清理 " + str(BASE))
except Exception as e:
    print(e)

print("\n✅ 00 完成。下一步按需跑 10/20/30/40 里的任意一格，最后跑 90 收产物。")

# 10 文生视频（Wan2.1 1.3B）

# 10 · 云端显卡出视频（Wan2.1 T2V 1.3B，T4 能跑）
为什么选它：开源、无 HF 门禁（不用申请授权）、1.3B 参数在 T4 16G 上 fp16 能跑 480p/49帧，
一条 5 秒片子约 2–4 分钟。要更高画质把 `PRESET` 换成 "quality"（显存不够会自己降级）。

In [ ]:
# 依赖：diffusers 0.33+ 才内置 Wan 管线
import subprocess, sys, os, json, time
from pathlib import Path

def sh(cmd, check=True):
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout.strip()[-3000:])
    if p.returncode != 0 and check:
        print(p.stderr.strip()[-2000:]); raise RuntimeError(cmd)
    return p

sh("pip -q install -U 'diffusers>=0.33' transformers accelerate ftfy sentencepiece imageio imageio-ffmpeg")

import torch
from diffusers import WanPipeline
from diffusers.utils import export_to_video

OUT = Path(os.environ.get("CK_OUT", "/content/colabkit/outputs/video"))
OUT.mkdir(parents=True, exist_ok=True)
print("输出目录:", OUT)

In [ ]:
# ── 载模型（第一次会下 ~5GB 到 HF 缓存；缓存在 Drive 上时下次秒起）──
MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("dtype:", dtype)

t0 = time.time()
pipe = WanPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
pipe.to("cuda")
pipe.set_progress_bar_config(disable=False)

# 显存吃紧时的省显存开关（T4 上先开着更稳，速度略降）
LOW_VRAM = True
if LOW_VRAM:
    pipe.enable_model_cpu_offload()
print(f"模型就绪 {time.time()-t0:.1f}s")

In [ ]:
# ── 生成 ──
PRESETS = {
    "draft":   dict(height=480, width=832, num_frames=33, num_inference_steps=25, guidance_scale=5.0),
    "standard":dict(height=480, width=832, num_frames=49, num_inference_steps=35, guidance_scale=5.0),
    "quality": dict(height=720, width=1280, num_frames=49, num_inference_steps=45, guidance_scale=5.5),
}

def make_video(prompt, preset="standard", negative=None, seed=None, name=None):
    cfg = dict(PRESETS[preset])
    g = torch.Generator(device="cpu")
    if seed is not None:
        g.manual_seed(int(seed))
    t = time.time()
    frames = pipe(
        prompt=prompt,
        negative_prompt=negative or "低质量, 模糊, 静止画面, 变形, 水印, 文字",
        generator=g,
        **cfg,
    ).frames[0]
    elapsed = time.time() - t
    stem = name or f"wan_{preset}_{int(time.time())}"
    mp4 = OUT / f"{stem}.mp4"
    export_to_video(frames, str(mp4), fps=16)
    size_mb = mp4.stat().st_size / 1024**2
    row = {"file": str(mp4), "prompt": prompt, "preset": preset, "seed": seed,
           "seconds": round(elapsed, 1), "size_mb": round(size_mb, 1), **cfg}
    print(json.dumps(row, ensure_ascii=False))
    return row

In [ ]:
# ── 跑一条（改 prompt 就行）──
rows = []
rows.append(make_video(
    "电影感镜头：雨夜便利店门口，一只橘猫蹲在自动贩卖机旁，霓虹招牌在积水里反光，镜头缓慢推近",
    preset="standard", seed=1234,
))

# 批量：一次排队多条，跑完统一记账
BATCH = [
    # ("another prompt", "draft"),
]
for i, (p, preset) in enumerate(BATCH):
    rows.append(make_video(p, preset=preset, seed=1000 + i))

(OUT / "manifest.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n✅ 完成 {len(rows)} 条，清单: {OUT/'manifest.json'}")
print("下载：左侧文件夹面板拖出来，或跑 90_push.py 打包。")

# 20 音乐 / 音效（MusicGen）

# 20 · 云端显卡出音乐 / 音效（MusicGen，稳、快、无门禁）
适合做视频配乐、氛围垫、十几秒的 loop。要更强的中文人声/翻唱，再上 ACE-Step（文件末尾给了装法）。
T4 上 musicgen-small 出一条 15 秒约 20–40 秒；medium 慢 3 倍但质感明显好。

In [ ]:
import subprocess, os, json, time
from pathlib import Path

def sh(cmd, check=True):
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout.strip()[-2000:])
    if p.returncode != 0 and check:
        print(p.stderr.strip()[-1500:]); raise RuntimeError(cmd)
    return p

sh("pip -q install -U transformers scipy soundfile")

import torch, numpy as np, scipy.io.wavfile as wavfile
from transformers import AutoProcessor, MusicgenForConditionalGeneration

OUT = Path(os.environ.get("CK_OUT", "/content/colabkit/outputs/music"))
OUT.mkdir(parents=True, exist_ok=True)

SIZE = os.environ.get("CK_MUSIC_SIZE", "medium")   # small / medium / large
MODEL = f"facebook/musicgen-{SIZE}"
print("模型:", MODEL)

processor = AutoProcessor.from_pretrained(MODEL)
model = MusicgenForConditionalGeneration.from_pretrained(
    MODEL, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to("cuda" if torch.cuda.is_available() else "cpu")
print("就绪")

In [ ]:
SR = model.config.audio_encoder.sampling_rate  # 32000
TOKENS_PER_SEC = 50

def make_music(prompt, seconds=15, seed=None, name=None, guidance=3.0):
    t = time.time()
    inputs = processor(text=[prompt], padding=True, return_tensors="pt").to(model.device)
    if seed is not None:
        torch.manual_seed(int(seed))
    with torch.no_grad():
        audio = model.generate(**inputs, do_sample=True, guidance_scale=guidance,
                               max_new_tokens=int(seconds * TOKENS_PER_SEC))
    arr = audio[0, 0].float().cpu().numpy()
    arr = np.clip(arr / max(1e-6, np.abs(arr).max()) * 0.95, -1, 1)
    stem = name or f"music_{int(time.time())}"
    f = OUT / f"{stem}.wav"
    wavfile.write(str(f), SR, (arr * 32767).astype(np.int16))
    row = {"file": str(f), "prompt": prompt, "seconds": seconds, "sr": SR,
           "elapsed": round(time.time() - t, 1), "size_mb": round(f.stat().st_size / 1024**2, 1)}
    print(json.dumps(row, ensure_ascii=False))
    return row

In [ ]:
rows = []
rows.append(make_music(
    "cinematic ambient piano with soft strings, slow build, warm analog texture, no drums",
    seconds=15, seed=7,
))
rows.append(make_music(
    "雨声 + 远处地铁轰鸣 + 便利店自动门叮咚，写实环境音，无音乐",
    seconds=10, seed=8,
))

# 和视频等长的一条背景垫（视频 5 秒 → 生成 6 秒留剪辑余量）
rows.append(make_music("minimal tech house loop, 100 bpm, filtered chords, subtle pump",
                       seconds=6, seed=9))

(OUT / "manifest.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n✅ {len(rows)} 条 → {OUT}")

## 想要「带人声 / 中文歌」就换 ACE-Step
```bash
!git clone --depth 1 https://github.com/ace-step/ACE-Step.git /content/ACE-Step
%cd /content/ACE-Step && pip -q install -e .
# 然后把模型缓存放 Drive： export ACESTEP_CACHE=/content/drive/MyDrive/colabkit/hf-cache/acestep
# 命令行出歌：
!acestep --config_path example.json --device cuda --output_path /content/colabkit/outputs/music
```
显存：T4 可用，但 3 分钟以上的整曲建议 A100/L4。

# 30 旁白 + SRT（edge-tts）

# 30 · 旁白（edge-tts：不占显存、秒出、带逐词时间轴）
为什么要它：GPU 那点时间应该留给视频/音乐；中文旁白用云端 TTS 质量已经够用，
而且它返回 **每个词的起止时间**，直接能生成字幕 SRT —— 做视频这步最省事。
想要「克隆你自己音色」，见文件末尾的 F5-TTS / CosyVoice 装法。

In [ ]:
import subprocess, os, json, asyncio, time
from pathlib import Path

def sh(cmd, check=True):
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout.strip()[-1500:])
    if p.returncode != 0 and check:
        print(p.stderr.strip()[-1200:]); raise RuntimeError(cmd)
    return p

sh("pip -q install -U edge-tts")

import edge_tts

OUT = Path(os.environ.get("CK_OUT", "/content/colabkit/outputs/voice"))
OUT.mkdir(parents=True, exist_ok=True)

VOICES = {
    "晓晓·女声·通用": "zh-CN-XiaoxiaoNeural",
    "云希·男声·年轻": "zh-CN-YunxiNeural",
    "云健·男声·解说": "zh-CN-YunjianNeural",
    "晓伊·女声·活泼": "zh-CN-XiaoyiNeural",
    "小北·东北女声": "zh-CN-liaoning-XiaobeiNeural",
    "英文·Ava": "en-US-AvaMultilingualNeural",
}

In [ ]:
def _ts(sec, sep=","):
    h = int(sec // 3600); m = int(sec % 3600 // 60); s = sec % 60
    return f"{h:02d}:{m:02d}:{int(s):02d}{sep}{int(round((s - int(s)) * 1000)):03d}"

async def _synth(text, voice, mp3, rate, pitch, volume):
    comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch, volume=volume)
    cues = []
    with open(mp3, "wb") as f:
        async for chunk in comm.stream():
            if chunk["type"] == "audio":
                f.write(chunk["data"])
            elif chunk["type"] == "WordBoundary":
                cues.append({"text": chunk["text"],
                             "start": chunk["offset"] / 1e7,
                             "end": (chunk["offset"] + chunk["duration"]) / 1e7})
    return cues

def tts(text, voice="zh-CN-XiaoxiaoNeural", name=None, rate="+0%", pitch="+0Hz", volume="+0%", chunk_chars=120):
    """长文本自动切片合成后拼接（edge-tts 单次太长会断），并合并时间轴。"""
    stem = name or f"voice_{int(time.time())}"
    mp3 = OUT / f"{stem}.mp3"
    all_cues, offset = [], 0.0
    parts = [text[i:i + chunk_chars] for i in range(0, len(text), chunk_chars)] or [""]
    files = []
    for i, part in enumerate(parts):
        seg = OUT / f".{stem}.part{i:03d}.mp3"
        cues = asyncio.run(_synth(part, voice, str(seg), rate, pitch, volume))
        dur = max([c["end"] for c in cues], default=0.0)
        for c in cues:
            all_cues.append({"text": c["text"], "start": c["start"] + offset, "end": c["end"] + offset})
        offset += dur
        files.append(seg)

    # 拼接
    lst = OUT / f".{stem}.txt"
    lst.write_text("\n".join(f"file '{f.as_posix()}'" for f in files), encoding="utf-8")
    sh(f'ffmpeg -y -loglevel error -f concat -safe 0 -i "{lst}" -c copy "{mp3}"')
    for f in files:
        f.unlink(missing_ok=True)
    lst.unlink(missing_ok=True)

    # 字幕（每 12 个字并成一条，读起来才不碎）
    srt = OUT / f"{stem}.srt"
    lines, idx, buf = [], 1, []
    for c in all_cues:
        buf.append(c)
        if sum(len(x["text"]) for x in buf) >= 12 or c is all_cues[-1]:
            s, e = buf[0]["start"], buf[-1]["end"]
            lines += [f"{idx}", f"{_ts(s)} --> {_ts(e)}", "".join(x["text"] for x in buf), ""]
            idx += 1
            buf = []

    srt.write_text("\n".join(lines), encoding="utf-8")
    dur = all_cues[-1]["end"] if all_cues else 0
    row = {"file": str(mp3), "srt": str(srt), "voice": voice, "chars": len(text),
           "duration_s": round(dur, 2), "size_mb": round(mp3.stat().st_size / 1024**2, 2)}
    print(json.dumps(row, ensure_ascii=False))
    return row

In [ ]:
rows = []
rows.append(tts(
    "想象一下，你站在一台自动贩卖机前面。投币，按下按钮，咚的一声，可乐滚出来。"
    "很长一段时间里，我们获取信息的方式，就跟这台售货机一模一样。",
    voice=VOICES["晓晓·女声·通用"], name="vending", rate="+6%",
))

(OUT / "manifest.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n✅ {OUT}")

## 要克隆音色（你自己的声音）
**F5-TTS**（中文效果好，T4 可跑）：
```bash
!pip -q install f5-tts
!f5-tts_infer-cli --model F5TTS_v1_Base \
  --ref_audio /content/drive/MyDrive/colabkit/ref_my_voice.wav \
  --ref_text "参考音频里说的那句话" \
  --gen_text "要合成的新台词" \
  --output_dir /content/colabkit/outputs/voice
```
**CosyVoice2**（更接近真人、支持方言，稍重）：官方仓库 FunAudioLLM/CosyVoice，需 `pip install -r requirements.txt`。
两者都要 3–10 秒干净人声做参考（无背景音乐、无混响）。

# 40 语音转文字（faster-whisper）

# 40 · 语音转文字 / 出字幕（faster-whisper large-v3，GPU）
用途：把已有视频/播客转成带时间轴的文稿；给成片烧字幕；把别人视频的内容扒下来做整理。
T4 上 large-v3 走 float16，1 小时音频约 4–8 分钟。

In [ ]:
import subprocess, os, json, time
from pathlib import Path

def sh(cmd, check=True):
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout.strip()[-1500:])
    if p.returncode != 0 and check:
        print(p.stderr.strip()[-1200:]); raise RuntimeError(cmd)
    return p

sh("pip -q install -U faster-whisper")

from faster_whisper import WhisperModel

OUT = Path(os.environ.get("CK_OUT", "/content/colabkit/outputs/asr"))
OUT.mkdir(parents=True, exist_ok=True)

MODEL_SIZE = os.environ.get("CK_WHISPER", "large-v3")   # large-v3 / medium / small
COMPUTE = "float16" if __import__("torch").cuda.is_available() else "int8"
print("模型:", MODEL_SIZE, COMPUTE)
model = WhisperModel(MODEL_SIZE, device="cuda" if COMPUTE != "int8" else "cpu", compute_type=COMPUTE)

In [ ]:
def _ts(sec):
    h = int(sec // 3600); m = int(sec % 3600 // 60); s = sec % 60
    return f"{h:02d}:{m:02d}:{int(s):02d},{int(round((s - int(s)) * 1000)):03d}"

def transcribe(media, language="zh", name=None, vad=True):
    media = Path(media)
    if not media.exists():
        # 视频先抽音轨（faster-whisper 自己能读视频，但先转 16k wav 更省内存）
        raise FileNotFoundError(media)
    t = time.time()
    segments, info = model.transcribe(
        str(media), language=language, beam_size=5,
        vad_filter=vad, vad_parameters={"min_silence_duration_ms": 400},
        word_timestamps=False,
    )
    stem = name or media.stem
    srt, txt, rows = [], [], []
    for i, s in enumerate(segments, 1):
        rows.append({"start": round(s.start, 2), "end": round(s.end, 2), "text": s.text.strip()})
        srt += [str(i), f"{_ts(s.start)} --> {_ts(s.end)}", s.text.strip(), ""]
        txt.append(s.text.strip())
    (OUT / f"{stem}.srt").write_text("\n".join(srt), encoding="utf-8")
    (OUT / f"{stem}.txt").write_text("\n".join(txt), encoding="utf-8")
    row = {"audio": str(media), "duration_s": round(info.duration, 1), "language": info.language,
           "segments": len(rows), "elapsed": round(time.time() - t, 1),
           "srt": str(OUT / f'{stem}.srt'), "txt": str(OUT / f'{stem}.txt')}
    print(json.dumps(row, ensure_ascii=False))
    return row, rows

In [ ]:
# 用法一：转 Drive 上的一个文件
# rows_info, segs = transcribe("/content/drive/MyDrive/colabkit/inputs/demo.mp4")

# 用法二：先用 yt-dlp 拉一个视频/音频再转（B 站的音频也行）
MEDIA_URL = os.environ.get("CK_MEDIA_URL", "")
if MEDIA_URL:
    sh("pip -q install -U yt-dlp", check=False)
    sh(f'cd /content && yt-dlp -x --audio-format m4a -o "/content/colabkit/in_%(id)s.%(ext)s" "{MEDIA_URL}"', check=False)
    cand = sorted(Path("/content/colabkit").glob("in_*"))
    if cand:
        transcribe(cand[-1])

print(f"\n✅ 输出目录 {OUT}")

# 50 Blender 云端渲染（Cycles + CUDA）

# 50 · Blender 云端渲染（Cycles + CUDA，T4 可用）
思路：
1. Blender 本体**缓存在 Drive**（只存 tar.xz），每次会话解压到 `/content`（快、不占 Drive 写入）
2. 渲染走 **Cycles GPU**：T4 支持 CUDA，第一次要先编译内核（1–3 分钟），之后就快了
3. 输出统一落 Drive，动画帧序列 + ffmpeg 合成 mp4

⚠️ EEVEE Next 需要 OpenGL 4.3 上下文，Colab 是无头环境，**大概率报错**；要 EEVEE 就自己加 `--gpu-backend egl` 试，别指望稳。

In [ ]:
import os, sys, json, glob, re, time, shutil, subprocess, urllib.request
from pathlib import Path

def sh(cmd, check=True, quiet=False, timeout=None):
    if not quiet:
        print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if p.stdout and not quiet:
        print(p.stdout.strip()[-3000:])
    if p.returncode != 0:
        print(p.stderr.strip()[-2000:])
        if check:
            raise RuntimeError(f"命令失败({p.returncode}): {cmd}")
    return p

DRIVE = Path("/content/drive/MyDrive")
BASE = (DRIVE / "colabkit") if DRIVE.exists() else Path("/content/colabkit")
APPS = BASE / "apps"
OUT = BASE / "outputs" / "blender"
for p in (APPS, OUT):
    p.mkdir(parents=True, exist_ok=True)
print("缓存:", APPS, "\n输出:", OUT)

In [ ]:
# ── 1. 装 Blender（缓存在 Drive，只下一次）──
BL_MIRROR = "https://download.blender.org/release"

def pick_latest(series="Blender4.5"):
    """从官方目录里挑最新的 linux-x64 tar.xz"""
    url = f"{BL_MIRROR}/{series}/"
    try:
        html = urllib.request.urlopen(url, timeout=30).read().decode("utf-8", "ignore")
    except Exception as e:
        print("列目录失败:", e)
        return None
    names = sorted(set(re.findall(rf"(blender-{series.replace('Blender','')}\.\d+-linux-x64\.tar\.xz)", html)))
    return names[-1] if names else None

def ensure_blender(series="Blender4.5", force=False):
    # 已经解压在 /content 里就直接用
    for cand in sorted(glob.glob("/content/blender-*/blender")):
        if not force:
            return cand
    name = pick_latest(series)
    if not name:
        raise RuntimeError("取不到 Blender 下载地址，检查网络或换 series")
    tarball = APPS / name
    if not tarball.exists() or force:
        print("下载", name, "（约 300 MB，只下一次）")
        urllib.request.urlretrieve(f"{BL_MIRROR}/{series}/{name}", tarball)
    print("解压到 /content …")
    sh(f'tar -xf "{tarball}" -C /content')
    cand = sorted(glob.glob("/content/blender-*/blender"))
    if not cand:
        raise RuntimeError("解压后没找到 blender 可执行文件")
    sh(f'chmod +x "{cand[-1]}"')
    print(sh(f'"{cand[-1]}" --version', quiet=True).stdout.split("\n")[0])
    return cand[-1]

BLENDER = ensure_blender()
print("BLENDER =", BLENDER)

In [ ]:
# ── 2. 渲染设置脚本（注入到任意 .blend）──
SETUP_PY = r'''
import bpy, sys, os
argv = sys.argv[sys.argv.index("--") + 1:] if "--" in sys.argv else []
opts = dict(a.split("=", 1) for a in argv if "=" in a)
samples = int(opts.get("samples", "32"))
rx, ry = opts.get("res", "960x540").split("x")
use_gpu = opts.get("gpu", "1") == "1"

sc = bpy.context.scene
sc.render.engine = "CYCLES"
sc.cycles.samples = samples
sc.cycles.use_denoising = True
sc.render.resolution_x, sc.render.resolution_y = int(rx), int(ry)
sc.render.resolution_percentage = 100
sc.render.image_settings.file_format = "PNG"

if use_gpu:
    try:
        prefs = bpy.context.preferences.addons["cycles"].preferences
        prefs.compute_device_type = "CUDA"
        prefs.get_devices()
        for d in prefs.devices:
            d.use = (d.type == "CUDA")
        sc.cycles.device = "GPU"
        print("DEVICES:", [(d.name, d.type, d.use) for d in prefs.devices])
    except Exception as e:
        print("GPU 设置失败，退回 CPU:", e)
        sc.cycles.device = "CPU"
else:
    sc.cycles.device = "CPU"
print("RENDER:", sc.render.engine, sc.cycles.device, samples, "samples", rx, "x", ry)
'''

def write_setup():
    p = "/content/blender_setup.py"
    Path(p).write_text(SETUP_PY, encoding="utf-8")
    return p

SETUP = write_setup()

In [ ]:
# ── 3. 冒烟测试：不需要任何 .blend，脚本现场建一个场景渲染一帧 ──
SMOKE_PY = r'''
import bpy, os, math, time
OUT = os.environ.get("SMOKE_OUT", "/content/blender_smoke.png")
t0 = time.time()

bpy.ops.wm.read_factory_settings(use_empty=True)
scene = bpy.context.scene

# 地面
bpy.ops.mesh.primitive_plane_add(size=12, location=(0, 0, 0))
floor = bpy.context.object
mat = bpy.data.materials.new("Floor")
mat.use_nodes = True
bsdf = mat.node_tree.nodes["Principled BSDF"]
bsdf.inputs["Base Color"].default_value = (0.05, 0.06, 0.08, 1)
bsdf.inputs["Roughness"].default_value = 0.35
floor.data.materials.append(mat)

# 主体：猴头 + 金属球 + 自发光环
bpy.ops.mesh.primitive_monkey_add(location=(-1.6, 0, 1.1), rotation=(0, 0, math.radians(20)))
monkey = bpy.context.object
for p in monkey.data.polygons:
    p.use_smooth = True
m = bpy.data.materials.new("Chrome")
m.use_nodes = True
b = m.node_tree.nodes["Principled BSDF"]
b.inputs["Base Color"].default_value = (0.9, 0.75, 0.35, 1)
b.inputs["Metallic"].default_value = 1.0
b.inputs["Roughness"].default_value = 0.15
monkey.data.materials.append(m)

bpy.ops.mesh.primitive_uv_sphere_add(radius=1.0, location=(1.4, 0.6, 1.0))
ball = bpy.context.object
ball.data.materials.append(m)

bpy.ops.mesh.primitive_torus_add(location=(0, -1.2, 2.2), major_radius=1.4, minor_radius=0.05)
ring = bpy.context.object
em = bpy.data.materials.new("Neon")
em.use_nodes = True
nt = em.node_tree
nt.nodes.clear()
out = nt.nodes.new("ShaderNodeOutputMaterial")
emi = nt.nodes.new("ShaderNodeEmission")
emi.inputs["Color"].default_value = (0.2, 0.8, 1.0, 1)
emi.inputs["Strength"].default_value = 12
nt.links.new(emi.outputs["Emission"], out.inputs["Surface"])
ring.data.materials.append(em)

# 灯光
bpy.ops.object.light_add(type="AREA", location=(3, -3, 6))
key = bpy.context.object
key.data.energy = 1200
key.data.size = 4
bpy.ops.object.light_add(type="POINT", location=(-4, 2, 3))
fill = bpy.context.object
fill.data.energy = 400
fill.data.color = (0.4, 0.7, 1.0)

# 相机
bpy.ops.object.camera_add(location=(6.5, -6.5, 4.2))
cam = bpy.context.object
cam.rotation_euler = (math.radians(66), 0, math.radians(45))
scene.camera = cam

# 世界：淡蓝灰
scene.world = bpy.data.worlds.new("W")
scene.world.use_nodes = True
bg = scene.world.node_tree.nodes["Background"]
bg.inputs["Color"].default_value = (0.06, 0.09, 0.12, 1)
bg.inputs["Strength"].default_value = 0.6

scene.render.filepath = OUT
scene.render.image_settings.file_format = "PNG"
bpy.ops.render.render(write_still=True)
print(f"SMOKE_OK {OUT} {time.time()-t0:.1f}s")
'''

def smoke_test(samples=32, res="960x540", use_gpu=True):
    p = Path("/content/blender_smoke.py")
    p.write_text(SMOKE_PY, encoding="utf-8")
    outp = "/content/blender_smoke.png"
    if os.path.exists(outp):
        os.remove(outp)
    cmd = f'"{BLENDER}" -b --factory-startup -P "{p}" -P "{SETUP}" -- samples={samples} res={res} gpu={1 if use_gpu else 0}'
    t0 = time.time()
    sh(cmd, quiet=True)
    ok = os.path.exists(outp)
    info = {"ok": ok, "file": outp if ok else None, "seconds": round(time.time() - t0, 1),
            "size_kb": round(os.path.getsize(outp) / 1024, 1) if ok else 0}
    print(json.dumps(info, ensure_ascii=False))
    return info

In [ ]:
smoke = smoke_test(samples=24)
if smoke["ok"]:
    shutil.copy(smoke["file"], OUT / "smoke.png")
    print("已复制到", OUT / "smoke.png")
    from IPython.display import Image, display
    display(Image(filename=str(OUT / "smoke.png")))

## 渲染你自己的 .blend
把 `.blend` 传到 Drive（建议 `MyDrive/colabkit/inputs/`），然后：

In [ ]:
def render_blend(blend, out_name=None, frames="1", samples=64, res="1280x720", use_gpu=True, fps=24):
    """渲染一段帧区间；frames 可写 '1'、'1-120'、'1,5,10'。返回输出目录。"""
    blend = Path(blend)
    if not blend.exists():
        raise FileNotFoundError(blend)
    out_dir = OUT / (out_name or blend.stem)
    out_dir.mkdir(parents=True, exist_ok=True)
    pat = str(out_dir / "frame_####")
    cmd = (f'"{BLENDER}" -b "{blend}" -P "{SETUP}" -- samples={samples} res={res} '
           f'gpu={1 if use_gpu else 0} -o "{pat}" -F PNG -f {frames}')
    t0 = time.time()
    sh(cmd, quiet=True)
    pngs = sorted(out_dir.glob("frame_*.png"))
    print(json.dumps({"frames": len(pngs), "dir": str(out_dir), "seconds": round(time.time() - t0, 1)}, ensure_ascii=False))
    return out_dir

def frames_to_mp4(frame_dir, fps=24, name=None):
    frame_dir = Path(frame_dir)
    pngs = sorted(frame_dir.glob("frame_*.png"))
    if not pngs:
        raise RuntimeError("没有帧可合成")
    mp4 = OUT / (name or f"{frame_dir.name}.mp4")
    sh(f'ffmpeg -y -loglevel error -framerate {fps} -pattern_type glob -i "{frame_dir}/frame_*.png" '
       f'-c:v libx264 -pix_fmt yuv420p -crf 18 "{mp4}"')
    print("mp4:", mp4, f"{mp4.stat().st_size/1024**2:.1f} MB")
    return mp4

# 示例（把路径换成你自己的）：
# d = render_blend("/content/drive/MyDrive/colabkit/inputs/my_scene.blend", frames="1-120", samples=64, res="1280x720")
# frames_to_mp4(d, fps=24)

## 常见问题
- **`无法分配后端`**：T4 池子满了。断开再连，等 1–2 分钟。设置已存成 T4，重连会自动申请。
- **第一次渲染很慢**：Cycles 要编译 CUDA 内核（1–3 分钟），第二次同会话就快。
- **`CUDA error: out of memory`**：降分辨率（`res="960x540"`）或降 `samples`，或改用 `use_gpu=False` 走 CPU。
- **EEVEE 报错**：无头环境没有 GL 上下文，用 Cycles；真要 EEVEE 试 `--gpu-backend egl`。
- **.blend 太大**：Drive 挂载读大文件慢，先 `cp` 到 `/content` 再渲染。
- **别在 `/content` 存成品**：会话一断就没了，统一落 `OUT`（Drive）。

In [ ]:
print("✅ 50 完成。Blender:", BLENDER)
print("输出目录:", OUT)

# 85 断线可续（检查点）

# 85 · 断线可续（检查点 + 续跑台账）
先说清楚：**没有哪段代码能延长官方给的时长**。免费实例的时长是按同时段需求动态给的
（现在常见 5 小时上下），到点就回收。能优化的是「断了之后恢复要多久」：
  1) 模型缓存放 Drive → 重连不用重下（00_env 已做）
  2) 产物实时落 Drive → 断线不丢成片（各 cell 的 OUT 已指向 Drive）
  3) 这一格维护一份 tasks.jsonl：跑完一条记一条，重连后只补没跑完的。
另外：**后台进程写文件不算「有活动」**，别指望它骗过闲置判定；真要长跑就保持标签页在前台。

In [ ]:
import json, os, time
from pathlib import Path

BASE = Path(os.environ.get("CK_BASE", "/content/colabkit"))
TASKS = BASE / "tasks.jsonl"
TASKS.parent.mkdir(parents=True, exist_ok=True)

def done_keys():
    if not TASKS.exists():
        return set()
    keys = set()
    for line in TASKS.read_text(encoding="utf-8").splitlines():
        try:
            r = json.loads(line)
            if r.get("status") == "done":
                keys.add(r["key"])
        except Exception:
            pass
    return keys

def mark(key, status="done", **extra):
    row = {"key": key, "status": status, "at": time.strftime("%Y-%m-%dT%H:%M:%S"), **extra}
    with open(TASKS, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("·", row["key"], row["status"])
    return row

def pending(all_keys):
    d = done_keys()
    return [k for k in all_keys if k not in d]

def resume_plan(all_keys):
    p = pending(all_keys)
    print(f"总共 {len(all_keys)} 条，已完成 {len(all_keys)-len(p)}，待跑 {len(p)}")
    for k in p:
        print("  待跑:", k)
    return p

In [ ]:
# 用法：把你要跑的活写成带唯一 key 的列表，重连后只补剩下的
JOBS = [
    # ("v1", "雨夜便利店 + 橘猫"),
    # ("v2", "太空电梯上升镜头"),
]
if JOBS:
    todo = resume_plan([k for k, _ in JOBS])
    for key, prompt in JOBS:
        if key not in todo:
            continue
        try:
            # row = make_video(prompt, preset="standard", name=key)   # ← 在这里调 10/20/30 里的函数
            mark(key, "done", prompt=prompt)
        except Exception as e:
            mark(key, "failed", prompt=prompt, error=str(e)[:300])
else:
    print("（JOBS 为空：先把要跑的活填进去）")

# 90 收产物（打包 / 落 Drive）

# 90 · 收产物（打包 / 落到 Drive / 出下载清单）
会话一断 `/content` 就清空，所以**每次跑完都执行这一格**。
有挂 Drive 时产物本来就在 Drive 里，这一步只是打包 + 出清单，方便手机端直接下载。

In [ ]:
import os, json, shutil, subprocess, time
from pathlib import Path

def sh(cmd, check=False):
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout.strip()[-2000:])
    if p.returncode != 0 and check:
        raise RuntimeError(cmd)
    return p

OUT_ROOT = Path(os.environ.get("CK_OUT_ROOT", "/content/colabkit/outputs"))
if not OUT_ROOT.exists():
    OUT_ROOT = Path("/content")
print("扫描:", OUT_ROOT)

files = [p for p in OUT_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in
         {".mp4", ".wav", ".mp3", ".png", ".jpg", ".srt", ".txt", ".json"}]
files.sort(key=lambda p: p.stat().st_mtime, reverse=True)

rows = [{"name": str(p.relative_to(OUT_ROOT)), "mb": round(p.stat().st_size / 1024**2, 2),
         "mtime": time.strftime("%H:%M:%S", time.localtime(p.stat().st_mtime))} for p in files]
for r in rows[:40]:
    print(f"  {r['mb']:>7.2f} MB  {r['mtime']}  {r['name']}")

print(f"\n共 {len(rows)} 个文件，合计 {sum(r['mb'] for r in rows):.1f} MB")

In [ ]:
# 打包（只打产物，跳过模型缓存）
if rows:
    stamp = time.strftime("%m%d-%H%M")
    zip_path = OUT_ROOT.parent / f"colabkit-{stamp}.zip"
    with shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(OUT_ROOT)) as _:
        pass
    print("📦", zip_path, f"{zip_path.stat().st_size/1024**2:.1f} MB")

# 若在 Drive 里，浏览器左侧「文件」面板就能下载；也可以在下一格出直链

In [ ]:
# 有 Drive 时的下载提示
DRIVE_ZIP = list(Path("/content/drive/MyDrive/colabkit").glob("*.zip")) if Path("/content/drive/MyDrive").exists() else []
for z in DRIVE_ZIP:
    print("Drive 里的压缩包（手机/电脑都能直接下）:", z)
print("\n✅ 收尾完成。接着可以：右上角「断开连接」省配额，或继续下一批。")